In [1]:
folder_path = "/Users/sahana/Documents/data_mining_db"

In [2]:
#Install required packages

import pandas as pd
import os
import unicodedata
import re
# !pip install langchain langchain-community langchain-groq sentence-transformers chromadb --break-system-packages
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import shutil

In [3]:
total_rows = 0

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(folder_path, file))
        rows = len(df)
        print(f"{file}: {rows} rows")
        total_rows += rows

print("Total rows in dataset:", total_rows)

Womens Innerwear.csv: 250 rows
Musical Instruments.csv: 250 rows
Mens Shirts.csv: 250 rows
Phones.csv: 170 rows
School Bags.csv: 250 rows
Refrigerators.csv: 250 rows
WomensFashion.csv: 250 rows
Mens Sports Shoes.csv: 250 rows
Womens Sandals.csv: 250 rows
Baby Products.csv: 250 rows
Toys and Games.csv: 250 rows
WesternWear.csv: 250 rows
Kids Clothing.csv: 250 rows
Motorbike Accessories.csv: 250 rows
Televisions.csv: 250 rows
Strength Training.csv: 250 rows
Mens Formal Shoes.csv: 250 rows
Handbags and Clutches.csv: 250 rows
Womens Watches.csv: 199 rows
Speakers.csv: 250 rows
Make-up.csv: 250 rows
Mens Innerwear.csv: 250 rows
Womens Shoes.csv: 250 rows
T-shirts and Polos.csv: 250 rows
Headphones.csv: 250 rows
Cameras.csv: 250 rows
Mens Watches.csv: 14 rows
Home Improvement.csv: 250 rows
Total rows in dataset: 6633


In [4]:
# Combining different CSV files from a data set into one single data frame.
df_list = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)

        df["source_file"] = file

        df_list.append(df)

# Combine all into one DataFrame
combined_df = pd.concat(df_list, ignore_index=True)

print("Total products after combining all product info from different csv files:", len(combined_df))
print()
print("Size of the data frame:", combined_df.shape)

Total products after combining all product info from different csv files: 6633

Size of the data frame: (6633, 7)


In [5]:
# Oitline of the dataset
combined_df.head()


,name,main_category,image,ratings,no_of_ratings,actual_price,source_file
0,Aston Andia Women Waist Cinchers Corset Adjust...,Womens Innerwear,https://m.media-amazon.com/images/I/51qLMr3Fea...,4.0,43.0,999.0,Womens Innerwear.csv
1,Floret Women's Cotton Non Padded Wire Free T-S...,Womens Innerwear,https://m.media-amazon.com/images/I/61sBzjDk7W...,4.0,1.0,399.0,Womens Innerwear.csv
2,DOT WAVE Women's Cotton Pyjamas - Loungepants ...,Womens Innerwear,https://m.media-amazon.com/images/I/81qQK0+n4p...,4.0,153.0,1699.0,Womens Innerwear.csv
3,Bali womens Comfort Revolution Wireless T-shir...,Womens Innerwear,https://m.media-amazon.com/images/I/81ReKuvZQr...,4.0,9328.0,1500.0,Womens Innerwear.csv
4,TRYLO Women's Non-Wired Bra,Womens Innerwear,https://m.media-amazon.com/images/W/IMAGERENDE...,4.0,11.0,435.0,Womens Innerwear.csv


In [6]:
# Checking for NaN values
combined_df.isnull().sum()


name             0
main_category    0
image            0
ratings          0
no_of_ratings    0
actual_price     0
source_file      0
dtype: int64

In [7]:
# Checking for empty or misiing values

print("Number of NaN values in each column:")
print(combined_df.isnull().sum())

print()
print("Number of missing values in each column:")
(combined_df == '').sum()

Number of NaN values in each column:
name             0
main_category    0
image            0
ratings          0
no_of_ratings    0
actual_price     0
source_file      0
dtype: int64

Number of missing values in each column:


name             0
main_category    0
image            0
ratings          0
no_of_ratings    0
actual_price     0
source_file      0
dtype: int64

In [8]:
# Checking for duplicate values and also dropping them

print("Total rows before dedup:", len(combined_df))

# print("Number of Duplicates present: ",combined_df.duplicated().sum())

combined_df[combined_df.duplicated(keep=False)]

combined_df = combined_df.drop_duplicates(subset=['name', 'main_category', 'actual_price'])

combined_df = combined_df.drop_duplicates(subset=['name']).reset_index(drop=True)

print("Total rows after dedup:", len(combined_df))

print("Number of Duplicates present after clean up: ",combined_df.duplicated().sum())

Total rows before dedup: 6633
Total rows after dedup: 6506
Number of Duplicates present after clean up:  0


In [9]:
print("Checking the data types of each column:")
print()
combined_df.info()

Checking the data types of each column:

<class 'pandas.DataFrame'>
RangeIndex: 6506 entries, 0 to 6505
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           6506 non-null   str    
 1   main_category  6506 non-null   str    
 2   image          6506 non-null   str    
 3   ratings        6506 non-null   float64
 4   no_of_ratings  6506 non-null   float64
 5   actual_price   6506 non-null   float64
 6   source_file    6506 non-null   str    
dtypes: float64(3), str(4)
memory usage: 355.9 KB


In [10]:
# Change the dtypr of the column no_of_ratings to int from float
combined_df['no_of_ratings'] = combined_df['no_of_ratings'].astype(int)

# Checking if all rating are valid
combined_df[(combined_df['ratings'] < 0) | (combined_df['ratings'] > 5)]
print("All the rating values are valid and are in the range 1 to 5.")
print()

# Change the dtypr of the column actual_price to int from float
combined_df['actual_price'] = combined_df['actual_price'].astype(int)

# Checking if the actual_price column has any -ve values
combined_df[combined_df['actual_price'] <= 0]
print("All the values in the column actual_price are valid, there is no 0 or -ve vales.")
print()

combined_df[["no_of_ratings", "actual_price"]].info()

USD_PER_INR = 0.011  # 1 INR ≈ 0.011 USD

combined_df['actual_price_usd'] = (combined_df['actual_price'] * USD_PER_INR).round(2)

All the rating values are valid and are in the range 1 to 5.

All the values in the column actual_price are valid, there is no 0 or -ve vales.

<class 'pandas.DataFrame'>
RangeIndex: 6506 entries, 0 to 6505
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   no_of_ratings  6506 non-null   int64
 1   actual_price   6506 non-null   int64
dtypes: int64(2)
memory usage: 101.8 KB


In [11]:
# creating ranking score 

combined_df["ranking_score"] = (
    combined_df["ratings"] * 0.7 +
    combined_df["no_of_ratings"].rank(pct=True) * 5 * 0.3
).round(2)

In [12]:
#  Standardizing Text Columns
combined_df['name'] = combined_df['name'].str.strip()

# Checking the categories in main_category
print("All values in column main_category:")
print()
print(combined_df['main_category'].unique())

# Changing the column name make-up to makeup
combined_df['main_category'] = combined_df['main_category'].replace({
    'make-up': 'makeup',
    't-shirts and polos': 'tshirts and polos',
})

All values in column main_category:

<StringArray>
[     'Womens Innerwear',   'Musical Instruments',           'Mens Shirts',
                'Phones',           'School Bags',         'Refrigerators',
        'Womens Fashion',     'Mens Sports Shoes',        'Womens Sandals',
         'Baby Products',                  'Toys',         'Kids Clothing',
 'Motorbike Accessories',           'Televisions',     'Strength Training',
     'Mens Formal Shoes', 'Handbags and Clutches',        'Womens Watches',
              'Speakers',               'Make-up',        'Mens Innerwear',
          'Womens Shoes',    'T-shirts and Polos',            'Headphones',
               'Cameras',          'Mens Watches',      'Home Improvement']
Length: 27, dtype: str


In [13]:
# Search corpus
# Reset index and create product_id
combined_df = combined_df.reset_index(drop=True)
combined_df['product_id'] = combined_df.index

# Create searchable text corpus with the new column search_text
combined_df['search_text'] = (
    'product: ' + combined_df['name'] + '. ' +
    'category: ' + combined_df['main_category'] + '. ' +
    'price: $' + combined_df['actual_price_usd'].astype(str) + '. ' +
    'rating: ' + combined_df['ratings'].astype(str) + ' out of 5. ' +
    'reviews: ' + combined_df['no_of_ratings'].astype(str) + '. ' +
    combined_df['name']
)

# Clean text for embeddings
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

combined_df['search_text'] = combined_df['search_text'].apply(clean_text)

print("Total number of columns: ",combined_df.columns.size)

print("Adter adding two new columns - product_id and search_text:")
print()
combined_df.head()

Total number of columns:  11
Adter adding two new columns - product_id and search_text:



,name,main_category,image,ratings,no_of_ratings,actual_price,source_file,actual_price_usd,ranking_score,product_id,search_text
0,Aston Andia Women Waist Cinchers Corset Adjust...,Womens Innerwear,https://m.media-amazon.com/images/I/51qLMr3Fea...,4.0,43,999,Womens Innerwear.csv,10.99,3.68,0,product: aston andia women waist cinchers cors...
1,Floret Women's Cotton Non Padded Wire Free T-S...,Womens Innerwear,https://m.media-amazon.com/images/I/61sBzjDk7W...,4.0,1,399,Womens Innerwear.csv,4.39,2.91,1,product: floret women's cotton non padded wire...
2,DOT WAVE Women's Cotton Pyjamas - Loungepants ...,Womens Innerwear,https://m.media-amazon.com/images/I/81qQK0+n4p...,4.0,153,1699,Womens Innerwear.csv,18.69,3.89,2,product: dot wave women's cotton pyjamas - lou...
3,Bali womens Comfort Revolution Wireless T-shir...,Womens Innerwear,https://m.media-amazon.com/images/I/81ReKuvZQr...,4.0,9328,1500,Womens Innerwear.csv,16.50,4.25,3,product: bali womens comfort revolution wirele...
4,TRYLO Women's Non-Wired Bra,Womens Innerwear,https://m.media-amazon.com/images/W/IMAGERENDE...,4.0,11,435,Womens Innerwear.csv,4.78,3.43,4,product: trylo women's non-wired bra. category...


In [14]:
# How sample serach_text looks for each row
print("Sample of how search_text lookds for each row/product: ")
print()
combined_df.iloc[0]["search_text"]

Sample of how search_text lookds for each row/product: 



'product: aston andia women waist cinchers corset adjustable hook&eye closure workout trainer flexible body shaper weight loss belt.... category: womens innerwear. price: $10.99. rating: 4.0 out of 5. reviews: 43. aston andia women waist cinchers corset adjustable hook&eye closure workout trainer flexible body shaper weight loss belt...'

In [15]:
# Cleaning text to create emdeding

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

combined_df['search_text'] = combined_df['search_text'].apply(clean_text)



In [16]:
combined_df.head()

,name,main_category,image,ratings,no_of_ratings,actual_price,source_file,actual_price_usd,ranking_score,product_id,search_text
0,Aston Andia Women Waist Cinchers Corset Adjust...,Womens Innerwear,https://m.media-amazon.com/images/I/51qLMr3Fea...,4.0,43,999,Womens Innerwear.csv,10.99,3.68,0,product: aston andia women waist cinchers cors...
1,Floret Women's Cotton Non Padded Wire Free T-S...,Womens Innerwear,https://m.media-amazon.com/images/I/61sBzjDk7W...,4.0,1,399,Womens Innerwear.csv,4.39,2.91,1,product: floret women's cotton non padded wire...
2,DOT WAVE Women's Cotton Pyjamas - Loungepants ...,Womens Innerwear,https://m.media-amazon.com/images/I/81qQK0+n4p...,4.0,153,1699,Womens Innerwear.csv,18.69,3.89,2,product: dot wave women's cotton pyjamas - lou...
3,Bali womens Comfort Revolution Wireless T-shir...,Womens Innerwear,https://m.media-amazon.com/images/I/81ReKuvZQr...,4.0,9328,1500,Womens Innerwear.csv,16.50,4.25,3,product: bali womens comfort revolution wirele...
4,TRYLO Women's Non-Wired Bra,Womens Innerwear,https://m.media-amazon.com/images/W/IMAGERENDE...,4.0,11,435,Womens Innerwear.csv,4.78,3.43,4,product: trylo women's non-wired bra. category...


In [17]:
#Remove special characters

combined_df['name'] = combined_df['name'].apply(lambda x: unicodedata.normalize('NFKD', str(x)))

In [18]:
# Checking the content lenght
print("Content Length of search_text column:")
print()
combined_df['search_text'].str.len().describe()

Content Length of search_text column:



count    6506.00000
mean      245.78174
std        69.15834
min        99.00000
25%       184.00000
50%       241.00000
75%       326.75000
max       346.00000
Name: search_text, dtype: float64

In [19]:
# Savinf this cleaned dataset
combined_df.to_csv("clean_data.csv", index=False)

In [20]:
# Checking the content lenght for search_text
print("Content Length of search_text column:")
print()
combined_df['search_text'].str.len().describe()

Content Length of search_text column:



count    6506.00000
mean      245.78174
std        69.15834
min        99.00000
25%       184.00000
50%       241.00000
75%       326.75000
max       346.00000
Name: search_text, dtype: float64

In [21]:
# Packaging the product data into a format LangChain — mainly for storing in Chroma and retrieving later

def build_doc(row):
    return Document(
        page_content=row['search_text'],
        metadata={
            "name": str(row["name"]),
            "category": str(row["main_category"]),
            "price_usd": float(row["actual_price_usd"]),
            "ratings": float(row["ratings"]),
        }
    )

docs = [build_doc(row) for _, row in combined_df.iterrows()]
print(f"Total documents: {len(docs)}")

Total documents: 6506


In [22]:
# import shutil
# import os

# # Delete stale vector store
# if os.path.exists("./product_db"):
#     shutil.rmtree("./product_db")
#     print("Deleted old product_db")

In [23]:
# Creating vector embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)


import shutil
import os

if os.path.exists("./product_db"):
    shutil.rmtree("./product_db")
    print("Deleted old product_db")

# Now create fresh vectorstore
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="./product_db"
)
print(f"Stored {vectorstore._collection.count()} products")
# Storing the embedding in Chroma DB
# vectorstore = Chroma.from_documents(
#     documents=docs,
#     embedding=embeddings,
#     persist_directory="./product_db"
# )
# print(f"Stored {vectorstore._collection.count()} products")

/var/folders/cm/0kxrpnjn4lq_92tmn6x2w41m0000gn/T/ipykernel_84634/718169233.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/opt/homebrew/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9139.27it/s]


Stored 6506 products


In [25]:
# Checking the pipeline
results = vectorstore.similarity_search("i am planning to go to beach")

for i, doc in enumerate(results[:3]):
    print(f"\n--- Document {i+1} ---")
    print("Name:", doc.metadata.get("name"))
    print("Category:", doc.metadata.get("category"))
    print("Price:", doc.metadata.get("price_usd"))
    print("Rating:", doc.metadata.get("ratings"))
    print("Content:", doc.page_content)


--- Document 1 ---
Name: Beach Toy Set for Kids 8 Pcs with Castle Shaped Bucket Shovels Water Sprinkler and Moulds Made in India Perfect Beach Toy ...
Category: Toys
Price: 3.58
Rating: 4.4
Content: product: beach toy set for kids 8 pcs with castle shaped bucket shovels water sprinkler and moulds made in india perfect beach toy .... category: toys. price: $3.58. rating: 4.4 out of 5. reviews: 13. beach toy set for kids 8 pcs with castle shaped bucket shovels water sprinkler and moulds made in india perfect beach toy ...

--- Document 2 ---
Name: BEREAL Macaron -White Cork Sandal Women
Category: Womens Sandals
Price: 14.29
Rating: 5.0
Content: product: bereal macaron -white cork sandal women. category: womens sandals. price: $14.29. rating: 5.0 out of 5. reviews: 1. bereal macaron -white cork sandal women

--- Document 3 ---
Name: Inc.5 womens 1191_peach Sandal
Category: Womens Sandals
Price: 27.39
Rating: 5.0
Content: product: inc.5 womens 1191_peach sandal. category: womens sandals. 

In [28]:
# Download the products stored as vector
shutil.make_archive("product_db", 'zip', "product_db")

'/Users/sahana/Documents/CMPE_255/rag_project/Intelligent-E-Commerce-Search-with-Retrieval-Augmented-Generation/product_db.zip'